# Lab | Agent & Vector store


<br>

## Intro

In this lab you'll build an AI agent that knows when to consult *different* knowledge bases to answer a question — instead of relying on a single source of truth.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: ingesting the *state of the union* speech and the *Ruff* docs into two vector stores, wrapping each in a `RetrievalQA` tool, and building an agent that picks the right tool (or both!) depending on the question.

2. **Replicate it yourself with a new dataset** — Then, you'll swap in a dataset of your choice and rebuild the same pipeline, adapting the prompts and tools along the way.

By the end of this lab, you'll understand how to build multi-source AI agents and be able to apply the pattern to your own datasets.

<br>

## Combine agents and vector stores

Let's get into the demo. We'll wrap each vector store in a `RetrievalQA` chain and hand it to an agent as a `Tool`. The agent then decides, at each step, which tool to call based purely on its description — this is what lets it route between multiple knowledge sources.

There are two flavors of this pattern, both of which we'll try below:

- **Agent as reasoner** — the agent calls a tool and can keep reasoning afterward (e.g. to combine results from multiple sources).
- **Agent as router** (`return_direct=True`) — the agent just picks the right tool and returns its answer immediately, no extra reasoning.

<br>

## Install dependencies

Uncomment and run the cells below to install the required dependencies for this notebook.

In [1]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

In [2]:
# !pip install python-dotenv==1.2.2 chromadb==1.5.9 beautifulsoup4==4.15.0

<br>

## Initial Setup

Before building anything, we need to load our API credentials, instantiate the LLM we'll use throughout the notebook, and locate the sample document we'll be querying.

In [3]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [4]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


In [5]:
llm = OpenAI(temperature=0, api_key=OPENAI_API_KEY)

<br>

Now let's ingest the state of the union speech: load the raw text, split it into manageable chunks, embed those chunks, and store them in a Chroma vector store.

In [6]:
doc_path =  "datasets/merchantofvenice.txt"

In [7]:
loader = TextLoader(doc_path, encoding="utf-8")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

docsearch = Chroma.from_documents(texts, embeddings, collection_name="state-of-union")

Created a chunk of size 1100, which is longer than the specified 1000
Created a chunk of size 1549, which is longer than the specified 1000
Created a chunk of size 2106, which is longer than the specified 1000
Created a chunk of size 1554, which is longer than the specified 1000
Created a chunk of size 1042, which is longer than the specified 1000
Created a chunk of size 1058, which is longer than the specified 1000
Created a chunk of size 1004, which is longer than the specified 1000
Created a chunk of size 1543, which is longer than the specified 1000
Created a chunk of size 1135, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 1202, which is longer than the specified 1000


In [8]:
merchantofvenice = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=docsearch.as_retriever()
)

<br>

## Adding a second knowledge source

To show how an agent can route between multiple tools, let's add a second vector store — this time built from the Ruff FAQ web page instead of a local file.

In [9]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [10]:
loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")

In [11]:
docs = loader.load()
ruff_texts = text_splitter.split_documents(docs)
ruff_db = Chroma.from_documents(ruff_texts, embeddings, collection_name="ruff")
ruff = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=ruff_db.as_retriever()
)

Created a chunk of size 2122, which is longer than the specified 1000
Created a chunk of size 3187, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2321, which is longer than the specified 1000


<br>

## Create the Agent

With both `RetrievalQA` chains ready, we wrap each one in a `Tool` (giving it a name and a description the agent will use to decide when to call it), then hand both tools to an agent.

In [12]:
# Import things that are needed generically
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_openai import OpenAI

<br>

Let's try it out — first with a question only the state of the union tool can answer, then one only Ruff can answer. Watch the verbose output to see which tool the agent picks each time.

In [13]:
tools = [
    Tool(
        name="Merchant of Venice QA System",
        func=merchantofvenice.run,
        description="useful for when you need to answer questions about the merchant of venice book. Input should be a fully formed question.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
    ),
]

In [14]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

C:\Users\kriti\AppData\Local\Temp\ipykernel_33308\1834837320.py:3: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


In [15]:
agent.invoke(
    "Who were the suitors to Portia?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should use the Merchant of Venice QA System to answer this question.
Action: Merchant of Venice QA System
Action Input: "Who were the suitors to Portia?"
Observation:  The suitors to Portia were the Neapolitan prince, the County Palatine, and Bassanio.
Thought: I now know the final answer.
Final Answer: The suitors to Portia were the Neapolitan prince, the County Palatine, and Bassanio.

> Finished chain.


{'input': 'Who were the suitors to Portia?',
 'output': 'The suitors to Portia were the Neapolitan prince, the County Palatine, and Bassanio.'}

In [16]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 You should consider the differences between ruff and flake8 before deciding which one to use.
Action: Ruff QA System
Action Input: "What are the differences between ruff and flake8?"
Observation:  Ruff has a larger rule set and does not support custom lint rules, while Flake8 supports plugins and allows for custom and third-party rules. Ruff also has a formatter and can automatically fix its own lint violations, while Flake8 does not have these capabilities. Additionally, Ruff is written in Rust while Flake8 is written in Python.
Thought: Now that I know the differences, I can make a more informed decision.
Final Answer: It ultimately depends on your specific needs and preferences, but some potential reasons to choose Ruff over Flake8 could include its larger rule set and automatic fixing capabilities.

> Finished chain.


{'input': 'Why use ruff over flake8?',
 'output': 'It ultimately depends on your specific needs and preferences, but some potential reasons to choose Ruff over Flake8 could include its larger rule set and automatic fixing capabilities.'}

## Use the Agent solely as a router

<br>

You can also set `return_direct=True` if you intend to use the agent as a router and just want to directly return the result of the RetrievalQAChain.

Notice that in the above examples the agent did some extra work after querying the RetrievalQAChain. You can avoid that and just return the result directly.

In [17]:
tools = [
    Tool(
        name="Merchant of Venice QA System",
        func=merchantofvenice.run,
        description="useful for when you need to answer questions about the merchant of venice book. Input should be a fully formed question.",
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question.",
        return_direct=True,
    ),
]

In [18]:
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [19]:
agent.invoke(
    "Who were the suitors to Portia?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should use the Merchant of Venice QA System to answer this question.
Action: Merchant of Venice QA System
Action Input: "Who were the suitors to Portia?"
Observation:  The suitors to Portia were the Neapolitan prince, the County Palatine, and Bassanio.


> Finished chain.


{'input': 'Who were the suitors to Portia?',
 'output': ' The suitors to Portia were the Neapolitan prince, the County Palatine, and Bassanio.'}

In [20]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 You should consider the differences between ruff and flake8 before deciding which one to use.
Action: Ruff QA System
Action Input: "What are the differences between ruff and flake8?"
Observation:  Ruff has a larger rule set and does not support custom lint rules, while Flake8 supports plugins and allows for custom and third-party rules. Ruff also has a formatter and can automatically fix its own lint violations, while Flake8 does not have these capabilities. Additionally, Ruff is written in Rust while Flake8 is written in Python.


> Finished chain.


{'input': 'Why use ruff over flake8?',
 'output': ' Ruff has a larger rule set and does not support custom lint rules, while Flake8 supports plugins and allows for custom and third-party rules. Ruff also has a formatter and can automatically fix its own lint violations, while Flake8 does not have these capabilities. Additionally, Ruff is written in Rust while Flake8 is written in Python.'}

In [21]:
agent.invoke(
    "What did biden say about ketanji brown jackson in the state of the union address?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should use the Merchant of Venice QA System since this question is about a political event.
Action: Merchant of Venice QA System
Action Input: "What did Biden say about Ketanji Brown Jackson in the State of the Union address?"
Observation:  I don't know, as this context is from a play by William Shakespeare and does not mention a State of the Union address or Biden.


> Finished chain.


{'input': 'What did biden say about ketanji brown jackson in the state of the union address?',
 'output': " I don't know, as this context is from a play by William Shakespeare and does not mention a State of the Union address or Biden."}

<br>

## Multi-Hop vector store reasoning

Because vector stores are easily usable as tools in agents, it is easy to use answer multi-hop questions that depend on vector stores using the existing agent framework.

In [22]:
tools = [
    Tool(
        name="Merchant of Venice QA System",
        func=merchantofvenice.run,
        description="useful for when you need to answer questions about the merchant of venice book. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
    Tool(
        name="Ruff QA System",
        func=ruff.run,
        description="useful for when you need to answer questions about ruff (a python linter). Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.",
    ),
]

In [23]:
# Construct the agent. We will use the default agent type here.
# See documentation for a full list of options.
agent = initialize_agent(
    tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True
)

In [24]:
agent.invoke(
    "Can ruff be used in VScode"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 Yes, ruff can be used in VScode
Action: Ruff QA System
Action Input: Can ruff be used in VScode
Observation:  Yes, Ruff can be used in VScode. It is available as a VScode extension and can be installed through the VScode marketplace.
Thought: I now know the final answer
Final Answer: Yes, ruff can be used in VScode.

> Finished chain.


{'input': 'Can ruff be used in VScode',
 'output': 'Yes, ruff can be used in VScode.'}

In [25]:
agent.invoke(
    "What tool does ruff use to run over Jupyter Notebooks? Was Bassanio a python coder and did he use Jupyter Notebooks?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should use the Ruff QA System to answer this question.
Action: Ruff QA System
Action Input: What tool does ruff use to run over Jupyter Notebooks? Was Bassanio a python coder and did he use Jupyter Notebooks?
Observation:  Ruff uses nbQA to run over Jupyter Notebooks. There is no information available about Bassanio's coding skills or use of Jupyter Notebooks.
Thought: I should use the Merchant of Venice QA System to answer the second part of the question.
Action: Merchant of Venice QA System
Action Input: Was Bassanio a python coder and did he use Jupyter Notebooks?
Observation:  I don't know.
Thought: I should try rephrasing the question to get more information.
Action: Merchant of Venice QA System
Action Input: Was Bassanio a programmer and did he use any coding tools?
Observation:  No, there is no indication in the context that Bassanio was a programmer or used any coding tools. He is described as a gentleman and is engaged in business ventures involving ships and trade, but the

{'input': 'What tool does ruff use to run over Jupyter Notebooks? Was Bassanio a python coder and did he use Jupyter Notebooks?',
 'output': "Ruff uses nbQA to run over Jupyter Notebooks, but there is no information available about Bassanio's coding skills or use of Jupyter Notebooks."}

In [26]:
agent.invoke(
    "Does Ruff support Python? Was Bassanio a python coder?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should check if Ruff supports Python first before answering the second question.
Action: Ruff QA System
Action Input: Does Ruff support Python?
Observation:  Yes, Ruff supports Python versions 3.7 and above.
Thought: Now that I know Ruff supports Python, I can answer the second question.
Action: Merchant of Venice QA System
Action Input: Was Bassanio a python coder?
Observation:  I don't know.
Thought: I should try rephrasing the question.
Action: Merchant of Venice QA System
Action Input: Was Bassanio a coder in Python?
Observation:  I don't know.
Thought: I should try asking about Bassanio's profession instead.
Action: Merchant of Venice QA System
Action Input: What was Bassanio's profession?
Observation:  Bassanio's profession was not explicitly stated in the given context. However, it can be inferred that he is a wealthy merchant, as he is able to borrow a large sum of money from Antonio and is also able to afford a trip to Belmont.
Thought: I now know the final answer.
Final An

{'input': 'Does Ruff support Python? Was Bassanio a python coder?',
 'output': "Bassanio's profession was not explicitly stated, but he is most likely a wealthy merchant."}

In [27]:
agent.invoke(
    "Does Ruff support Python? Who did Portia marry?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should use the Ruff QA System to answer the first question and the Merchant of Venice QA System to answer the second question.
Action: Ruff QA System
Action Input: Does Ruff support Python?
Observation:  Yes, Ruff supports Python versions 3.7 and above.
Thought: Now I should use the Merchant of Venice QA System to answer the second question.
Action: Merchant of Venice QA System
Action Input: Who did Portia marry?
Observation:  Bassanio
Thought: I now know the final answer.
Final Answer: Bassanio

> Finished chain.


{'input': 'Does Ruff support Python? Who did Portia marry?',
 'output': 'Bassanio'}

In [28]:
agent.invoke(
    "Did Shylock know about Pylint? What is the difference between Ruff and Pylint?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should use the Ruff QA System to answer this question since it is about Pylint.
Action: Ruff QA System
Action Input: What is the difference between Ruff and Pylint?
Observation:  Ruff and Pylint are both code quality tools for Python, but they have different approaches and capabilities. Ruff is a linter, which means it checks for code style and potential errors, while Pylint is a type checker, which means it checks for type errors and potential bugs. Ruff has a larger rule set and can automatically fix some lint violations, but it does not support custom or third-party rules like Pylint does. Additionally, Ruff is designed to be used alongside a type checker, while Pylint can be used on its own.
Thought: Now I need to find out if Shylock knew about Pylint.
Action: Merchant of Venice QA System
Action Input: Did Shylock know about Pylint?
Observation:  No, Shylock did not know about Pylint.
Thought: I now know the final answer.
Final Answer: No, Shylock did not know about Pylint. The 

{'input': 'Did Shylock know about Pylint? What is the difference between Ruff and Pylint?',
 'output': 'No, Shylock did not know about Pylint. The difference between Ruff and Pylint is that Ruff is a linter and Pylint is a type checker.'}

In [29]:
agent.invoke(
    "What is 2+2?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 This is a simple math question that can be easily answered.
Action: None
Action Input: None
Observation: None is not a valid tool, try one of [Merchant of Venice QA System, Ruff QA System].
Thought: I should use a math calculator to solve this question.
Action: None
Action Input: None
Observation: None is not a valid tool, try one of [Merchant of Venice QA System, Ruff QA System].
Thought: I should use a math library to solve this question.
Action: None
Action Input: None
Observation: None is not a valid tool, try one of [Merchant of Venice QA System, Ruff QA System].
Thought: I should use a math function to solve this question.
Action: None
Action Input: None
Observation: None is not a valid tool, try one of [Merchant of Venice QA System, Ruff QA System].
Thought: I should use a calculator tool to solve this question.
Action: Merchant of Venice QA System
Action Input: "What is 2+2?"
Observation:  I don't know.
Thought: I should try a different tool.
Action: Ruff QA System
Action Inpu

{'input': 'What is 2+2?',
 'output': 'None of the tools provided can answer the question "What is 2+2?" as it is not related to the context provided.'}